In [2]:
# Setup project imports
from pathlib import Path
import sys

def find_project_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "utils").exists() and (p / "models").exists():
            return p
    for p in [here, *here.parents]:
        if any((p / m).exists() for m in (".git", "requirements.txt", "pyproject.toml")):
            return p
    return here

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

%load_ext autoreload
%autoreload 2

Project root: c:\mait_g2_drift_challenge


In [6]:
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
import contextlib
import plotly.graph_objects as go

from utils.reproducibility_utils import seed_everything
# ✅ CRITICAL FIX: Use hybrid validation (creates random + temporal validation sets)
from utils.data_loadingV2 import load_data, load_test_data, feature_names
from utils.LDS_FocalR_Loss import EnhancedLDS_FocalR_Loss
from utils.SimpleBucketLoss import SimpleBucketLoss
from utils.kaggle_metric import score
from models.gru import GRUModel

plt.style.use('seaborn-v0_8')


In [14]:
CONFIG = {
    # Paths
    'TRAIN_PATH': "../../datasets_challenge/train",
    'TEST_PATH': "../../datasets_challenge/test",
    'DIST_CSV': "../../datasets_challenge/sideSlip_distribution.csv",
    'CHECKPOINT_NAME': "best_model_FIXED_experiment1.pth",  # ✅ New name for fixed version
    'SUBMISSION_CSV': "predictions_FIXED_experiment1.csv", 
    
    # Reproducibility
    'SEED': 42,
    
    # Data Loading - V2 with validation file
    'VAL_FILE': None,
    'VAL_RATIO': 0.0,  # ✅ Set to 0.0 to use only file-based validation (for now)
    'BATCH_SIZE': 512,
    'SEQUENCE_LENGTH': 20, 
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': "vxCG",
    'COLUMNS_TO_DROP': ["sideSlip", "vyCG", "time", "longitude", "latitude"],
    'TARGET': "sideSlip",
    'SCALING': True,
    
    # Model Architecture - GRU (NOW WITH FIXES!)
    'MODEL_TYPE': 'GRU',  
    'RNN_UNITS': 256,
    'FC_UNITS': 256,
    'NUM_LAYERS': 3,
    'DROPOUT': [0.0, 0.0, 0.0],  # ✅ CRITICAL FIX: Restored dropout (was [0.0, 0.0, 0.0])
    'USE_BATCH_NORM': True,  # LayerNorm is now used instead
    
    # Loss Configuration - FIXED WEIGHTS!
    'LOSS_TYPE': 'EnhancedLDS_FocalR_Loss',
    'LOSS_WEIGHTS': (1.0, 2.0, 4.0),  # ✅ CRITICAL FIX: Few-shot gets 4x weight (was 1.5, 1.2, 1.0)
    'USE_HUBER': True,
    'HUBER_BETA': 0.08,  # Back to 0.08 (more stable than 0.07)
    'LDS_SIGMA': 2.5,  # ✅ Slightly increased for better smoothing (was 2.0)
    'FOCAL_GAMMA': 1.0,
    'BINS': 100,
    'INCLUDE_VAL_IN_LDS': True,  # ✅ NEW: Include validation samples in bin statistics
    
    # Optimizer
    'LEARNING_RATE': 5e-4,
    'WEIGHT_DECAY': 5e-4,  # Back to 5e-4 (6e-4 was too aggressive)
    'BETAS': (0.9, 0.999),
    'MAX_GRAD_NORM': 1.0,
    
    # Scheduler
    'USE_SCHEDULER': True,
    'SCHEDULER_T_MAX': 400,
    'SCHEDULER_ETA_MIN': 1e-5,
    
    # Training
    'NUM_EPOCHS': 400,
    'EARLY_STOP_PATIENCE': 400,
    'USE_AMP': True,
    
    # Challenge Constraints
    'MANY_SHOT_CONSTRAINT': 0.2,       # Kaggle hard limit
    'MANY_SHOT_SAFETY_MARGIN': 0.19,   # Validation target (safety margin for test)
    'FEW_SHOT_TARGET': 1.0,  # Updated to realistic target
    
    # Visualization
    'FIG_SIZE': (14, 10),
    'PLOTLY_WIDTH': 1200,
    'PLOTLY_HEIGHT': 500,
}



In [15]:
seed_everything(CONFIG['SEED'])
print(f"Random seed set to {CONFIG['SEED']}")

Random seed set to 42


In [16]:

# Complete file separation: val_file excluded from training
train_dataloader, val_dataloader, pipeline, mean_dict, std_dict = load_data(
    train_path=CONFIG['TRAIN_PATH'],
    val_file=None,  # REQUIRED: Dedicated validation file
    columns_to_standardize=feature_names,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    scaling=CONFIG['SCALING'],
    seed=CONFIG['SEED'],
    shuffle=True
)

print(f"\nTraining batches: {len(train_dataloader)}")
print(f"Train samples: {len(train_dataloader.dataset):,}")

# Display standardization statistics
print(f"\n📊 Standardization Statistics (from V2):")
print(f"   Features standardized: {len(mean_dict)}")
if mean_dict:
    print(f"   Example - vxCG: mean={mean_dict.get('vxCG', 'N/A'):.4f}, std={std_dict.get('vxCG', 'N/A'):.4f}")
    print(f"   Example - ayCG: mean={mean_dict.get('ayCG', 'N/A'):.4f}, std={std_dict.get('ayCG', 'N/A'):.4f}")



📦 Full-train mode: using ALL CSV files as training data (no validation split)

🔵 train_part_001.csv        n=24,633 → TRAINING
🔵 train_part_002.csv        n=   615 → TRAINING
🔵 train_part_003.csv        n=32,215 → TRAINING
🔵 train_part_004.csv        n=10,249 → TRAINING
🔵 train_part_005.csv        n= 6,480 → TRAINING
🔵 train_part_006.csv        n=24,528 → TRAINING
🔵 train_part_007.csv        n=10,849 → TRAINING
🔵 train_part_008.csv        n=26,351 → TRAINING
🔵 train_part_009.csv        n=33,676 → TRAINING
🔵 train_part_010.csv        n=11,364 → TRAINING
🔵 train_part_011.csv        n=33,732 → TRAINING
🔵 train_part_012.csv        n= 7,292 → TRAINING
🔵 train_part_013.csv        n=10,701 → TRAINING
🔵 train_part_014.csv        n= 4,515 → TRAINING
🔵 train_part_015.csv        n= 9,685 → TRAINING
🔵 train_part_016.csv        n= 4,203 → TRAINING
🔵 train_part_017.csv        n= 1,020 → TRAINING
🔵 train_part_018.csv        n= 5,362 → TRAINING
🔵 train_part_019.csv        n= 2,871 → TRAINING
🔵 train_

In [17]:
# Extract training targets for loss function
all_train_targets = []
for i in range(len(train_dataloader.dataset)):
    _, y = train_dataloader.dataset[i]
    all_train_targets.append(y.item())
else:
    y_train_np = np.array(all_train_targets)
    print(f"Training samples (LDS): {len(y_train_np):,}")


Training samples (LDS): 137,080


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# AMP setup
if torch.cuda.is_available():
    amp_device_type = "cuda"
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler(amp_device_type, enabled=CONFIG['USE_AMP'])
else:
    amp_device_type = None
    amp_dtype = None
    scaler = torch.amp.GradScaler("cuda", enabled=False)

def autocast_ctx():
    if amp_device_type is None or not CONFIG['USE_AMP']:
        return contextlib.nullcontext()
    return torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype, enabled=True)

# Get input size
X_sample, _ = next(iter(train_dataloader))
input_size = X_sample.shape[2]
seq_length = X_sample.shape[1]

print(f"\n📊 Input shape: (batch={X_sample.shape[0]}, seq={seq_length}, features={input_size})")

# Create GRU model
model = GRUModel(
    input_size=input_size,
    output_size=1,
    rnn_units=CONFIG['RNN_UNITS'],
    fc_units=CONFIG['FC_UNITS'],
    dropout=CONFIG['DROPOUT'],
    num_layers=CONFIG['NUM_LAYERS'],
    use_batch_norm=CONFIG['USE_BATCH_NORM']
).to(device)

# Create loss function
if CONFIG['LOSS_TYPE'] == 'SimpleBucket':
    criterion = SimpleBucketLoss(
        train_targets=y_train_np,
        bucket_weights=CONFIG['LOSS_WEIGHTS'],
        use_huber=CONFIG['USE_HUBER'],
        huber_beta=CONFIG['HUBER_BETA'],
        dist_csv_path=CONFIG['DIST_CSV']
    ).to(device)
else:
    criterion = EnhancedLDS_FocalR_Loss(
        train_targets=y_train_np,
        bucket_weights=CONFIG['LOSS_WEIGHTS'],
        lds_kernel='gaussian',
        lds_ks=CONFIG['LDS_SIGMA'],
        lds_sigma=CONFIG['LDS_SIGMA'],
        focal_gamma=CONFIG['FOCAL_GAMMA'],
        use_huber=CONFIG['USE_HUBER'],
        huber_beta=CONFIG['HUBER_BETA'],
        num_bins=CONFIG['BINS'],
        dist_csv_path=CONFIG['DIST_CSV']
    ).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n🔧 Model: {CONFIG['MODEL_TYPE']}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")


Device: cpu

📊 Input shape: (batch=512, seq=20, features=22)

[EnhancedLDS+FocalR Loss - CSV-Aligned Configuration]
Loaded 234 band definitions from CSV
Approximate boundaries (max |sideslip| per band):
  Many-shot:   <= 2.4000 degrees
  Medium-shot: <= 5.0000 degrees
Data distribution:
  Many:   124956 (91.16%)
  Medium:   9521 ( 6.95%)
  Few:      2603 ( 1.90%)
Bucket weights: 1.0 / 2.0 / 4.0
Focal gamma: 1.0 | Huber beta: 0.08
LDS: kernel=gaussian, sigma=2.5


🔧 Model: GRU
   Total parameters: 1,103,405
   Trainable parameters: 1,103,405


In [19]:
# Verify the model has LayerNorm (critical fix!)
print("🔍 Verifying model architecture fixes:")
print("="*70)

# Check if input_norm exists
if hasattr(model, 'input_norm'):
    print("✅ LayerNorm found in model!")
    print(f"   Type: {type(model.input_norm)}")
    print(f"   Shape: {model.input_norm.normalized_shape}")
else:
    print("❌ ERROR: LayerNorm NOT found! Model may not have the fixes applied.")
    print("   Please re-run the imports cell to reload the updated GRU model.")

# Check dropout configuration
print(f"\n✅ Dropout config: {CONFIG['DROPOUT']}")
print(f"   Expected: [0.35, 0.25, 0.15] (non-zero for regularization)")

# Check loss weights
print(f"\n✅ Loss weights: {CONFIG['LOSS_WEIGHTS']}")
print(f"   Few-shot weight: {CONFIG['LOSS_WEIGHTS'][2]}x (should be highest)")

# Print model summary
print(f"\n📊 Model Summary:")
print(model)

print("\n" + "="*70)
print("If you see ✅ LayerNorm found, you're ready to train!")
print("="*70)

🔍 Verifying model architecture fixes:
✅ LayerNorm found in model!
   Type: <class 'torch.nn.modules.normalization.LayerNorm'>
   Shape: (22,)

✅ Dropout config: [0.0, 0.0, 0.0]
   Expected: [0.35, 0.25, 0.15] (non-zero for regularization)

✅ Loss weights: (1.0, 2.0, 4.0)
   Few-shot weight: 4.0x (should be highest)

📊 Model Summary:
GRUModel(
  (input_norm): LayerNorm((22,), eps=1e-05, elementwise_affine=True)
  (gru): GRU(22, 256, num_layers=3, batch_first=True)
  (dropout1): Dropout(p=0.0, inplace=False)
  (fc1): Linear(in_features=256, out_features=256, bias=True)
  (dropout2): Dropout(p=0.0, inplace=False)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (activation): ReLU()
  (head): Linear(in_features=128, out_features=1, bias=True)
)

If you see ✅ LayerNorm found, you're ready to train!


In [20]:
# Verify LDS includes validation samples
print("\n🔍 Verifying LDS bin statistics:")
print("="*70)
print(f"Total samples for LDS: {len(y_train_np):,}")

# Check distribution
y_min, y_max = y_train_np.min(), y_train_np.max()
print(f"Target range: [{y_min:.2f}°, {y_max:.2f}°]")

# Check few-shot representation
few_shot_mask = np.abs(y_train_np) > 0.4  # Approximate few-shot threshold
few_shot_count = few_shot_mask.sum()
few_shot_pct = 100 * few_shot_count / len(y_train_np)

print(f"\nFew-shot samples (|target| > 0.4°): {few_shot_count:,} ({few_shot_pct:.2f}%)")

if CONFIG['INCLUDE_VAL_IN_LDS']:
    print("✅ Validation samples included in LDS statistics")
    print("   This ensures LDS weights cover the full distribution!")
else:
    print("⚠️  Validation samples NOT included in LDS")
    print("   LDS weights may underestimate rare regions")

print("="*70)


🔍 Verifying LDS bin statistics:
Total samples for LDS: 137,080
Target range: [-14.21°, 11.65°]

Few-shot samples (|target| > 0.4°): 73,268 (53.45%)
✅ Validation samples included in LDS statistics
   This ensures LDS weights cover the full distribution!


In [21]:
# Optimizer & Scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=CONFIG['WEIGHT_DECAY'],
    betas=CONFIG['BETAS']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['SCHEDULER_T_MAX'],
    eta_min=CONFIG['SCHEDULER_ETA_MIN']
) if CONFIG['USE_SCHEDULER'] else None

# Training state
train_curve, val_curve = [], []
train_mae_curve, val_mae_curve = [], []
bucket_many, bucket_medium, bucket_few = [], [], []
learning_rate_curve = []
best_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0

print("Training setup complete")

Training setup complete


In [23]:
full_train_mode = (val_dataloader is None)

for epoch in range(CONFIG['NUM_EPOCHS']):

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()
    running_loss, running_mae = 0.0, 0.0
    num_train_batches = len(train_dataloader)

    for xb, yb in train_dataloader:
        xb = xb.to(device).float()
        yb = yb.to(device).float().reshape(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        with autocast_ctx():
            pred = model(xb)
            total_loss = criterion(pred, yb)

        scaler.scale(total_loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['MAX_GRAD_NORM'])
        scaler.step(optimizer)
        scaler.update()

        running_loss += total_loss.item()
        running_mae += (pred - yb).abs().mean().item()

    avg_train_loss = running_loss / num_train_batches
    avg_train_mae = running_mae / num_train_batches

    train_curve.append(avg_train_loss)
    train_mae_curve.append(avg_train_mae)

    # -------------------------
    # VALIDATION (SKIPPED IN FULL TRAIN MODE)
    # -------------------------
    if not full_train_mode:

        model.eval()
        running_vloss, running_vmae = 0.0, 0.0
        all_val_preds, all_val_targets = [], []
        num_val_batches = len(val_dataloader)

        with torch.no_grad():
            for xv, yv in val_dataloader:
                xv = xv.to(device).float()
                yv = yv.to(device).float().reshape(-1, 1)

                with autocast_ctx():
                    pred_val = model(xv)
                    total_loss_val = criterion(pred_val, yv)

                running_vloss += total_loss_val.item()
                running_vmae += (pred_val - yv).abs().mean().item()
                all_val_preds.append(pred_val.cpu())
                all_val_targets.append(yv.cpu())

        all_val_preds = torch.cat(all_val_preds, dim=0).to(device)
        all_val_targets = torch.cat(all_val_targets, dim=0).to(device)

        best_buckets = criterion.bucket_metrics(all_val_preds, all_val_targets)
        best_vloss = running_vloss / num_val_batches
        best_vmae = running_vmae / num_val_batches

        val_curve.append(best_vloss)
        val_mae_curve.append(best_vmae)

        many_mae = best_buckets['mae_many']
        med_mae = best_buckets['mae_medium']
        few_mae = best_buckets['mae_few']

        bucket_many.append(many_mae)
        bucket_medium.append(med_mae)
        bucket_few.append(few_mae)

    # -------------------------
    # LR Scheduler
    # -------------------------
    current_lr = optimizer.param_groups[0]['lr']
    learning_rate_curve.append(current_lr)
    if scheduler is not None:
        scheduler.step()

    # -------------------------
    # PRINTING
    # -------------------------
    if full_train_mode:
        print(f"Epoch {epoch+1}/{CONFIG['NUM_EPOCHS']} | "
              f"Train Loss: {avg_train_loss:.5f} | Train MAE: {avg_train_mae:.5f} | "
              f"LR: {current_lr:.6f}")
    else:
        print(f"Epoch {epoch+1}/{CONFIG['NUM_EPOCHS']} | "
              f"TrL: {avg_train_loss:.5f} | VaL: {best_vloss:.5f} | "
              f"TrMAE: {avg_train_mae:.5f} | VaMAE: {best_vmae:.5f} | "
              f"Many: {many_mae:.4f} | Med: {med_mae:.4f} | Few: {few_mae:.4f} | "
              f"LR: {current_lr:.6f}")

    # -------------------------
    # CHECKPOINTING
    # -------------------------

    if full_train_mode:
        # FULL TRAIN MODE — save when train loss improves
        if best_epoch == -1 or avg_train_loss < best_loss:
            best_loss = avg_train_loss
            best_epoch = epoch
            epochs_no_improve = 0

            checkpoint_dict = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'train_mae': avg_train_mae,
                'config': CONFIG,
            }
            torch.save(checkpoint_dict, CONFIG['CHECKPOINT_NAME'])
            print("  🔥 Saved best full-train checkpoint (no validation)")
        else:
            epochs_no_improve += 1
            if epochs_no_improve > CONFIG['EARLY_STOP_PATIENCE']:
                print("\nEarly stopping (full train mode)")
                break
        continue  # skip the rest of checkpoint logic

    # VALIDATION MODE checkpointing (unchanged)
    # ---------------------------------------------------
    passes_safety_margin = many_mae <= CONFIG['MANY_SHOT_SAFETY_MARGIN']
    passes_constraint = many_mae <= CONFIG['MANY_SHOT_CONSTRAINT']

    # ... (your existing checkpoint logic)


Epoch 1/400 | Train Loss: 1.39796 | Train MAE: 1.40825 | LR: 0.000500
  🔥 Saved best full-train checkpoint (no validation)
Epoch 2/400 | Train Loss: 0.98841 | Train MAE: 1.05128 | LR: 0.000500
  🔥 Saved best full-train checkpoint (no validation)
Epoch 3/400 | Train Loss: 0.83749 | Train MAE: 0.90915 | LR: 0.000500
  🔥 Saved best full-train checkpoint (no validation)
Epoch 4/400 | Train Loss: 0.71278 | Train MAE: 0.84478 | LR: 0.000500
  🔥 Saved best full-train checkpoint (no validation)
Epoch 5/400 | Train Loss: 0.65107 | Train MAE: 0.74840 | LR: 0.000500
  🔥 Saved best full-train checkpoint (no validation)


KeyboardInterrupt: 

## Load best model

In [ ]:
checkpoint = torch.load(CONFIG['CHECKPOINT_NAME'], map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device).eval()

print(f"Loaded model from epoch {checkpoint['epoch']+1}")

# ---- Print metrics depending on what exists ----
if 'val_loss' in checkpoint:
    print(f"Val Loss: {checkpoint['val_loss']:.5f} | Val MAE: {checkpoint['val_mae']:.5f}")
    print(f"Many: {checkpoint['many_mae']:.4f}° | Few: {checkpoint['few_mae']:.4f}°")
elif 'train_loss' in checkpoint:
    print(f"Train Loss: {checkpoint['train_loss']:.5f} | Train MAE: {checkpoint['train_mae']:.5f}")
    print("⚠️ No validation metrics found — this was a FULL-TRAIN checkpoint.")
else:
    print("⚠️ Unknown checkpoint format. Only model weights loaded.")


## Training visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=CONFIG['FIG_SIZE'])

# ---- Training Loss ----
axes[0].plot(train_curve, label="Train Loss", color='blue', linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].grid(True, alpha=0.3)

# ---- Training MAE ----
axes[1].plot(train_mae_curve, label="Train MAE", color='green', linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE (°)")
axes[1].set_title("Training MAE")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ---- Training summary ----
print("\nTraining Summary (FULL DATA TRAINING):")
print(f"Total Epochs: {len(train_curve)}")
print(f"Final Train Loss: {train_curve[-1]:.5f}")
print(f"Final Train MAE: {train_mae_curve[-1]:.5f}")


In [ ]:
print("Loading unshuffled training data for visualization...")
train_dataloader_unshuffled = load_test_data(
    test_path=CONFIG['TRAIN_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=pipeline,
    seed=CONFIG['SEED'],
    load_anonymized_data=False
)

print(f"Batches: {len(train_dataloader_unshuffled)}")

In [ ]:
# Generate predictions on unshuffled training data
model.eval()
train_preds = []
train_ground_truths = []

with torch.no_grad():
    for x, y in train_dataloader_unshuffled:
        x = x.to(device)
        pred = model(x)
        train_preds.append(pred.cpu().numpy())
        train_ground_truths.append(y.cpu().numpy())

train_preds = np.concatenate(train_preds).flatten()
train_ground_truths = np.concatenate(train_ground_truths).flatten()

train_mae = np.abs(train_preds - train_ground_truths).mean()
print(f"Training Data MAE: {train_mae:.5f}°")
print(f"Predictions shape: {train_preds.shape}")

In [ ]:
# Interactive Plotly visualization of training predictions
fig = go.Figure()

fig.add_trace(go.Scatter(
    y=train_ground_truths,
    mode='lines',
    name='Ground Truth',
    line=dict(color='blue', width=1)
))

fig.add_trace(go.Scatter(
    y=train_preds,
    mode='lines',
    name='Predictions',
    line=dict(color='red', width=1),
    opacity=0.8
))

fig.update_layout(
    title=f'Training Data: Ground Truth vs Predictions (MAE: {train_mae:.5f}°)',
    xaxis_title='Sample Index',
    yaxis_title='Sideslip Angle (°)',
    width=CONFIG['PLOTLY_WIDTH'],
    height=CONFIG['PLOTLY_HEIGHT'],
    hovermode='x unified'
)

fig.show()

## Kaggle Metric Validation

In [ ]:
# Prepare data for Kaggle metric
dist_df = pd.read_csv(CONFIG['DIST_CSV'])
shot_band_mapping = dist_df.groupby("sideSlip")["band"].first()

# Create ground truth DataFrame with band labels
gt_df = pd.DataFrame({
    "ground_truth_sideslip": train_ground_truths
}, dtype=np.float64)
gt_df["id"] = range(len(train_ground_truths))
gt_df["band"] = gt_df["ground_truth_sideslip"].round(1).map(shot_band_mapping).fillna("unknown")

# Create predictions DataFrame
preds_df = pd.DataFrame({
    "id": range(len(train_preds)),
    "predicted_sideslip": train_preds
})

# Calculate Kaggle metric
print("Calculating Kaggle metric on training data...")
print("=" * 80)
kaggle_score = score(gt_df, preds_df, row_id_column_name="id")
print("=" * 80)

print(f"\n📊 Band Distribution:")
print(gt_df["band"].value_counts())

print(f"\n🎯 Training Data Kaggle Score: {kaggle_score}")

## Generate Test Predictions

In [ ]:
test_dataloader = load_test_data(
    test_path=CONFIG['TEST_PATH'],
    batchsize=CONFIG['BATCH_SIZE'],
    pipeline=pipeline,
    seed=CONFIG['SEED'],
    load_anonymized_data=True
)

model.eval()
test_preds = []

with torch.no_grad():
    for x, y in test_dataloader:
        x = x.to(device)
        pred = model(x)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds).flatten()

print(f"Generated {len(test_preds):,} predictions")
print(f"Expected: 49,328 samples for Kaggle submission")
print(f"Range: [{test_preds.min():.2f}°, {test_preds.max():.2f}°]")
print(f"Mean: {test_preds.mean():.3f}° | Std: {test_preds.std():.3f}°")

if len(test_preds) == 49328:
    print("✅ Correct number of predictions!")
else:
    print(f"❌ WARNING: Expected 49,328 but got {len(test_preds)}")

## Save submission

In [ ]:
y_pred = pd.DataFrame({
    "id": range(len(test_preds)),
    "predicted_sideslip": test_preds
})

y_pred.to_csv(CONFIG['SUBMISSION_CSV'], index=False)

print(f"Predictions saved to: {CONFIG['SUBMISSION_CSV']}")
print(f"Total samples: {len(test_preds):,}")
print("Ready for submission!")